<a href="https://colab.research.google.com/github/KaykyTajima/cifake-xai-detection/blob/feat%2Ftreinamento-all-cnn/notebooks/nome_do_seu_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Treinamento da Rede

In [ ]:
# 0. INSTALAÇÃO DE DEPENDÊNCIAS E IMPORTAÇÃO DE BIBLIOTECAS
!pip install -q datasets

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks, metrics

# Verificação de GPU no Google Colab
print("Dispositivos GPU disponíveis:", tf.config.list_physical_devices('GPU'))

Dispositivos GPU disponíveis: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# 1.1 Carregar o dataset na memória via Hugging Face
print("\nBaixando o dataset CIFAKE do Hugging Face (Isso pode levar alguns minutos)...")
dataset = load_dataset("dragonintelligence/CIFAKE-image-dataset")

# 1.2 Definir diretórios de destino no ambiente do Colab (/content)
base_dir = "/content/data/raw"
train_dir = f"{base_dir}/train"
test_dir = f"{base_dir}/test"
weights_dir = "/content/models/saved_weights"

os.makedirs(weights_dir, exist_ok=True)

# 1.3 Função para estruturar e salvar as imagens nas pastas corretas
def export_images_to_local(hf_dataset, split_name):
    print(f"\nIniciando exportação estruturada do conjunto de {split_name.upper()}...")
    for i, item in enumerate(tqdm(hf_dataset[split_name])):
        image = item['image']
        label = item['label']

        # Mapeamento estrito da classe: 0 = real (CIFAR-10), 1 = fake (Stable Diffusion)
        class_name = "real" if label == 0 else "fake"

        # Cria a hierarquia: /content/data/raw/train/real, /content/data/raw/train/fake, etc.
        save_dir = os.path.join(base_dir, split_name, class_name)
        os.makedirs(save_dir, exist_ok=True)

        # Salva a imagem em PNG para preservar intactos os micro-artefatos de difusão
        image_path = os.path.join(save_dir, f"{class_name}_{i}.png")
        image.save(image_path)

# Executa a extração
export_images_to_local(dataset, 'train')
export_images_to_local(dataset, 'test')
print("\nAs 120.000 imagens foram exportadas com sucesso para /content/data/raw/!")

# 1.4 Carregar amostra de imagens com OpenCV e converter de BGR para RGB
caminho_real = os.path.join(train_dir, 'real')
caminho_fake = os.path.join(train_dir, 'fake')

arquivos_reais = [os.path.join(caminho_real, f) for f in os.listdir(caminho_real) if f.endswith('.png')][:5]
arquivos_fakes = [os.path.join(caminho_fake, f) for f in os.listdir(caminho_fake) if f.endswith('.png')][:5]

def carregar_imagem_cv2(caminho):
    img_bgr = cv2.imread(caminho)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb

reais = [carregar_imagem_cv2(arq) for arq in arquivos_reais]
fakes = [carregar_imagem_cv2(arq) for arq in arquivos_fakes]


Baixando o dataset CIFAKE do Hugging Face (Isso pode levar alguns minutos)...

Iniciando exportação estruturada do conjunto de TRAIN...


  0%|          | 0/100000 [00:00<?, ?it/s]


Iniciando exportação estruturada do conjunto de TEST...


  0%|          | 0/20000 [00:00<?, ?it/s]


As 120.000 imagens foram exportadas com sucesso para /content/data/raw/!


In [ ]:
# 2.1 Carregar o Dataset (Treino e Teste) via API do Keras
print("\nCarregando o dataset de Treino...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='binary',      # Classificação binária (0 = Real, 1 = Sintética)
    class_names=['real', 'fake'],
    color_mode='rgb',
    batch_size=32,
    image_size=(32, 32),      # Dimensões originais preservadas para não destruir micro-ruídos
    shuffle=True
)
print("\nCarregando o dataset de Teste...")
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='binary',
    class_names=['real', 'fake'],
    color_mode='rgb',
    batch_size=32,
    image_size=(32, 32),
    shuffle=False
)


Carregando o dataset de Treino...
Found 100000 files belonging to 2 classes.

Carregando o dataset de Teste...
Found 20000 files belonging to 2 classes.


### ⚙️ Pré-processamento: Normalização Estatística e Restrições Geométricas

A etapa de pré-processamento das imagens no conjunto CIFAKE foi delimitada para focar na extração de assinaturas matemáticas, descartando deliberadamente práticas convencionais de visão computacional.

**1. Normalização Estatística**
A única transformação aplicada aos dados é a normalização dos canais de cor RGB para uma escala de distribuição estatística com média zero e variância unitária. Essa padronização é indispensável para mitigar disparidades de iluminação entre as amostras. Matematicamente, se os pixels originais (que variam de 0 a 255) não fossem centralizados no zero, os pixels mais luminosos teriam uma força de gradiente desproporcional e dominariam a atualização dos pesos da rede. A normalização simetriza as forças de atualização, permitindo que a rede capture o "piso de ruído" tanto nas áreas iluminadas quanto nas sombras, sem cair no viés de associar brilho global à classe sintética.

**2. Por que NÃO aplicar Redimensionamento ?**
O dataset possui uma resolução espacial nativa de 32x32 pixels, a qual deve ser integralmente preservada. A aplicação de qualquer redimensionamento exigiria interpolações matemáticas que destruiriam imediatamente os micro-artefatos espectrais e os ruídos de alta frequência inerentes ao processo de amostragem da difusão. Ao restringir as dimensões espaciais, obriga-se a rede neural a buscar anomalias estatísticas locais em vez de focar em anomalias semânticas macroscópicas.

**3. Por que NÃO aplicamos Ampliação Artificial ?**
Técnicas agressivas de ampliação artificial de dados, como distorções geométricas, rotações ou cortes espaciais, não foram utilizadas no escopo da pesquisa. O descarte do *data augmentation* assegura que as assinaturas espectrais originais deixadas pelo *Stable Diffusion* permaneçam puras e estruturalmente intactas para a leitura da arquitetura convolucional.

In [ ]:
# 2.2 Definir e Aplicar Normalização Estatística
def standardize_image(image, label):
    standardized_image = tf.image.per_image_standardization(image)
    return standardized_image, label

print("Aplicando normalização estatística (média zero, variância unitária)...")
normalized_train_dataset = train_dataset.map(standardize_image)
normalized_test_dataset = test_dataset.map(standardize_image)

# Comprovação da ingestão e normalização
for images_norm, labels in normalized_train_dataset.take(1):
    batch_norm = images_norm.numpy()
    print("\n--- Estatísticas do Primeiro Lote (Treino Normalizado) ---")
    print(f"Formato do Tensor: {batch_norm.shape}")
    print(f"Média do Lote: {np.mean(batch_norm):.4f}")
    print(f"Desvio Padrão do Lote: {np.std(batch_norm):.4f}")

Aplicando normalização estatística (média zero, variância unitária)...

--- Estatísticas do Primeiro Lote (Treino Normalizado) ---
Formato do Tensor: (32, 32, 32, 3)
Média do Lote: -0.0000
Desvio Padrão do Lote: 1.0000


### 🧠 Construção e Compilação da Arquitetura All-CNN-C

Abaixo está o detalhamento estrutural do código que instancia e treina a arquitetura **All Convolutional Net (All-CNN-C)**, adaptada para a classificação binária de imagens geradas por difusão.

*   **Inicialização (`he_normal`):** Os pesos dos filtros são inicializados com a distribuição de He, o que é matematicamente necessário para redes profundas baseadas na ativação não linear `ReLU`.
    *   **Comparativo Matemático:** A inicialização padrão do Keras (Xavier/Glorot) assume ativações lineares ou simétricas e define a variância dos pesos iniciais como $\text{Variância}(W) = \frac{1}{n_{\text{in}}}$ (onde $n_{\text{in}}$ é o número de conexões de entrada do filtro). Contudo, a função `ReLU` zera todos os valores negativos, cortando a variância do sinal pela metade a cada camada, o que leva ao desvanecimento dos gradientes.
    *   **A Solução de He:** Para compensar essa perda exata de 50%, a fórmula de He dobra a variância original: $\text{Variância}(W) = \frac{2}{n_{\text{in}}}$. Assim, os pesos são sorteados de uma distribuição Normal com média $0$ e desvio padrão $\sigma = \sqrt{\frac{2}{n_{\text{in}}}}$. Isso garante que a magnitude do sinal não desapareça ou exploda ao atravessar a rede.
*   **Bloco 1 (Extração e 1ª Subamostragem):**
    *   Inicia recebendo o tensor na resolução nativa de $32\times32$ pixels em 3 canais de cor (RGB).
    *   Aplica duas convoluções padrão de 96 filtros ($3\times3$) para capturar padrões locais.
    *   **Abolição do Max-Pooling:** Em vez de descartar informações através do agrupamento máximo tradicional, a redução espacial de $32\times32$ para $16\times16$ é realizada ativamente por uma camada convolucional operando com passo de subamostragem (`strides=(2, 2)`). Isso permite que a rede aprenda a reduzir a imagem preservando os micro-ruídos originais.
    *   Aplica-se uma taxa de descarte (`Dropout` de $0.5$) para inibir o sobreajuste (*overfitting*).
*   **Bloco 2 (Abstração e 2ª Subamostragem):**
    *   Dobra a profundidade para 192 filtros ($3\times3$) e aplica uma segunda convolução com `strides=(2, 2)`, reduzindo a topografia espacial de $16\times16$ para $8\times8$. Aplica-se novamente o `Dropout` de $0.5$.
*   **Bloco 3 (Convoluções $1\times1$ e Projeção Binária):**
    *   Substitui as tradicionais e volumosas camadas densas por convoluções de filtro $1\times1$. Isso refina os canais de forma independente em cada pixel, sem misturar vizinhanças.
    *   A camada `conv_class` reduz os 192 mapas de características para apenas **1 unidade de saída** com ativação linear (`activation=None`), perfeitamente desenhada para emitir os *logits* da classificação estritamente binária definida na injeção do dataset (`label_mode='binary'`).
*   **Classificação Terminal (GAP e Sigmoide):**
    *   O **Global Average Pooling (GAP)** calcula a média aritmética espacial da grade de $8\times8$, traduzindo as respostas dos filtros sem destruir a localidade das características.
    *   A função de ativação **Sigmoide** converte a saída única em uma distribuição de probabilidade contínua entre $0.0$ (Real) e $1.0$ (Sintética).
*   **Compilação e Otimização:**
    *   O modelo é compilado com o otimizador Adam (`learning_rate=3e-4`) e a função de perda **Binary Crossentropy** (Entropia Cruzada Binária), ideal para saídas baseadas em uma unidade sigmoide.
    *   As métricas monitoradas refletem rigorosamente a metodologia do projeto: **Acurácia**, **Precisão** e **Recall (Sensibilidade)**.
*   **Pipeline de Treinamento (Callbacks e Prefetch):**
    *   O treinamento é controlado por mecanismos dinâmicos: salvamento exclusivo dos melhores pesos (`ModelCheckpoint`), interrupção prematura em caso de estagnação (`EarlyStopping`) e decaimento adaptativo da taxa de aprendizado (`ReduceLROnPlateau`). A injeção de dados ocorre de forma assíncrona utilizando `prefetch` para maximizar o uso da GPU.

In [ ]:
def build_all_cnn_c(input_shape=(32, 32, 3), num_classes=2):
    model = models.Sequential(name="All-CNN-C")
    model.add(layers.InputLayer(shape=input_shape))

    # Inicializador ideal para camadas com ativação ReLU
    init = 'he_normal'

    # --- BLOCO 1 ---
    model.add(layers.Conv2D(96, (3, 3), padding='same', activation='relu', kernel_initializer=init, name='conv1_1'))
    model.add(layers.Conv2D(96, (3, 3), padding='same', activation='relu', kernel_initializer=init, name='conv1_2'))
    model.add(layers.Conv2D(96, (3, 3), strides=(2, 2), padding='same', activation='relu', kernel_initializer=init, name='conv1_stride'))
    model.add(layers.Dropout(0.5, name='dropout_1'))

    # --- BLOCO 2 ---
    model.add(layers.Conv2D(192, (3, 3), padding='same', activation='relu', kernel_initializer=init, name='conv2_1'))
    model.add(layers.Conv2D(192, (3, 3), padding='same', activation='relu', kernel_initializer=init, name='conv2_2'))
    model.add(layers.Conv2D(192, (3, 3), strides=(2, 2), padding='same', activation='relu', kernel_initializer=init, name='conv2_stride'))
    model.add(layers.Dropout(0.5, name='dropout_2'))

    # --- BLOCO 3 ---
    model.add(layers.Conv2D(192, (3, 3), padding='same', activation='relu', kernel_initializer=init, name='conv3_1'))
    # Refinamento dos pesos aprendidos pixel a pixel
    model.add(layers.Conv2D(192, (1, 1), padding='valid', activation='relu', kernel_initializer=init, name='conv3_1x1'))
    # Substituição ao Flatten, comprimindo para um único mapa de caracteristicas(tensor de ordem 2 = matriz)
    model.add(layers.Conv2D(1, (1, 1), padding='valid', activation=None, kernel_initializer=init, name='conv_class'))

    # --- CLASSIFICAÇÃO TERMINAL ---
    model.add(layers.GlobalAveragePooling2D(name='gap'))
    model.add(layers.Activation('sigmoid', name='predictions'))

    return model

In [ ]:
# Instanciar o modelo
model = build_all_cnn_c()

In [ ]:
# Compilar modelo
model.compile(
    optimizer=optimizers.Adam(learning_rate=3e-4),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall')
    ]
)

In [ ]:
# Definir callbacks para etapas de regularização, cronograma de lr e salvar os melhores pesos
checkpoint_cb = callbacks.ModelCheckpoint(
    filepath=os.path.join(weights_dir, 'all_cnn_c_best.keras'),
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping_cb = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler_cb = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
# Otimizar a execução da GPU
AUTOTUNE = tf.data.AUTOTUNE
# Garantir execução de normalização de estatística enquanto GPU treina os lotes
train_ds = normalized_train_dataset.prefetch(buffer_size=AUTOTUNE)
val_ds = normalized_test_dataset.prefetch(buffer_size=AUTOTUNE)

EPOCAS = 25

print("\nIniciando o treinamento da All-CNN-C no dataset CIFAKE...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCAS,
    callbacks=[checkpoint_cb, early_stopping_cb, lr_scheduler_cb]
)


Iniciando o treinamento da All-CNN-C no dataset CIFAKE...
Epoch 1/25
3124/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7306 - loss: 0.5113 - precision: 0.7331 - recall: 0.7134
Epoch 1: val_loss improved from None to 0.23744, saving model to /content/models/saved_weights/all_cnn_c_best.keras

Epoch 1: finished saving model to /content/models/saved_weights/all_cnn_c_best.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 68s 18ms/step - accuracy: 0.8200 - loss: 0.3866 - precision: 0.8216 - recall: 0.8175 - val_accuracy: 0.9062 - val_loss: 0.2374 - val_precision: 0.8717 - val_recall: 0.9526 - learning_rate: 3.0000e-04
Epoch 2/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9086 - loss: 0.2297 - precision: 0.9091 - recall: 0.9076
Epoch 2: val_loss improved from 0.23744 to 0.18775, saving model to /content/models/saved_weights/all_cnn_c_best.keras

Epoch 2: finished saving model to /content/models/saved_weights/all_cnn_c_best.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 53s 17ms/step - accu

### ⚙️ Ajuste de Hiperparâmetros e Exploração Arquitetural (*Model Tuning*)

Com a convergência inicial bem-sucedida do modelo base, a próxima etapa consiste em otimizar o comportamento dinâmico e testar a robustez da rede através do refinamento sistemático de seus hiperparâmetros operacionais. Para evitar custos computacionais proibitivos, realizar-se-á uma busca em grade (*Grid Search*) em uma amostra representativa do conjunto de treinamento. O espaço de busca concentrará esforços nos seguintes eixos da arquitetura:

**1. Dinâmica de Atualização dos Pesos:**
*   **Taxa de Aprendizado Inicial (*Learning Rate*):** Avaliação do tamanho do passo inicial ótimo para a descida do gradiente (ex: `1e-3`, `5e-4`, `1e-4`). A descoberta da taxa inicial adequada é fundamental para garantir que a rede convirja de forma estável, podendo atuar em conjunto com mecanismos de decaimento da taxa de aprendizado (*Learning Rate Schedulers*) previstos na metodologia.
*   **Tamanho do Lote (*Batch Size*):** Exploração do volume de processamento simultâneo (ex: `16`, `32`, `64`). Lotes menores introduzem ruído estocástico regularizador, ao passo que lotes maiores estabilizam as estimativas matemáticas do gradiente durante o treinamento.

**2. Exploração de Otimizadores Alternativos:**
*   Exploração de otimizadores SGD tradicional, Adam e Nadam( RMSProp + Acelerador de Nesterov )

**3. Substituição da Função de Ativação:**
*   Exploração de funções de ativação Relu e LeakyRelu(utilizar a função de inicialização default do TensorFlow) com objetivo de evitar o "DyingRelu".